# exp4 — YOLOv11n

Mesma configuração do `exp2` (dataset version 2, 100 épocas, imgsz 640, batch 16, seed 42) — a única variável trocada é a arquitetura do modelo (`yolov8n.pt` → `yolo11n.pt`). Não retreina o exp2, que você já tem pronto.

In [ ]:
!pip install -q ultralytics roboflow torch

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="SUA_API_KEY_AQUI")  # use Colab Secrets / variável de ambiente, nunca hardcoded no notebook salvo
project = rf.workspace("pedros-workspace-1w4rz").project("my-first-project-kfo4l")

# MESMA versão usada no exp2 (dataset original, não o fork aumentado
# usado no exp3) — é isso que garante que a única variável mudando
# entre exp2 e exp4 seja a arquitetura do modelo.
version = project.version(2)
dataset = version.download("yolov8")

print(project.versions())

In [ ]:
import torch

def selecionar_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = selecionar_device()
device

In [ ]:
# ---- Guarda de integridade de dados ----
# O projeto já teve um incidente real de mixup exp2/exp3 (Seção 7.1 do
# relatório). Este assert confirma que o dataset baixado é mesmo a
# version 2 (a mesma do exp2) antes de gastar 100 épocas de GPU
# treinando com o dataset errado.
expected_version_marker = "-2"  # Roboflow inclui a versão no nome da pasta exportada
assert expected_version_marker in dataset.location, (
    f"ERRO DE INTEGRIDADE: esperava a dataset version 2 (mesma do exp2), "
    f"mas o path baixado foi '{dataset.location}'. Pare e confira antes de treinar."
)
print(f"✅ Dataset confirmado: {dataset.location}")

In [ ]:
from ultralytics import YOLO

# Única mudança real em relação ao exp2: a arquitetura do modelo.
model = YOLO('yolo11n.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,          # igual ao exp2
    imgsz=640,           # igual ao exp2
    batch=16,            # igual ao exp2
    device=device,
    seed=42,             # igual ao exp2 — reprodutibilidade
    project='runs', name='exp4',
)

print("NB COMPLETED SUCCESSFULLY — READY FOR field validation on exp4")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
shutil.make_archive('exp4_zipped', 'zip', 'runs/exp4')
print('Folder "runs/exp4" successfully zipped to "exp4_zipped.zip"')

In [ ]:
from google.colab import files
files.download('exp4_zipped.zip')